# Day 5: Indexing Practice with EXPLAIN ANALYZE

## Objective
Learn how database indexes work in practice by measuring query performance before and after creating indexes. You will:

1. Use `EXPLAIN ANALYZE` to see exactly how PostgreSQL executes queries
2. Create indexes and observe the speedup
3. Test indexes on JOINs and composite indexes
4. Understand when PostgreSQL's optimizer *chooses not to use* an index (and why that's correct)
5. Drop indexes when they are no longer needed

## Table of Contents
1. [Connection Setup](#connection-setup)
2. [Sequential Scan — No Index](#1-sequential-scan--no-index)
3. [Create an Index and Compare](#2-create-an-index-and-compare)
4. [Indexing Foreign Keys for JOINs](#3-indexing-foreign-keys-for-joins)
5. [Composite Indexes](#4-composite-indexes)
6. [When the Optimizer Ignores Your Index](#5-when-the-optimizer-ignores-your-index)
7. [Dropping Indexes](#6-dropping-indexes)
8. [Try It Yourself Exercises](#7-try-it-yourself-exercises)

## Connection Setup

We use the same `run_query` helper from previous days for **queries that return rows** (SELECT). It connects to the PostgreSQL database, executes a query, returns results as a pandas DataFrame, and always closes the connection.

In this notebook we also run **commands that return no rows** — `CREATE INDEX` and `DROP INDEX` (these are DDL, *Data Definition Language*). A DataFrame helper can't handle those (there are no rows to put in a DataFrame!), and DDL changes must be **committed** to take effect. So we add a second helper, `run_command`, for exactly that.

**Rule of thumb:** `run_query` for SELECT, `run_command` for everything else (CREATE / DROP / INSERT / UPDATE / DELETE).

In [1]:
import psycopg2
import pandas as pd

def run_query(sql: str) -> pd.DataFrame:
    """Run a SELECT query and return results as a DataFrame."""
    conn = psycopg2.connect(
        host="localhost", port=5432,
        dbname="week2_db", user="student", password="student123"
    )
    try:
        df = pd.read_sql_query(sql, conn)
        return df
    finally:
        conn.close()

def run_command(sql: str) -> None:
    """Run a SQL command that returns no rows (CREATE INDEX, DROP INDEX, ...).

    Unlike run_query, this commits the transaction so the change is saved.
    """
    conn = psycopg2.connect(
        host="localhost", port=5432,
        dbname="week2_db", user="student", password="student123"
    )
    try:
        with conn.cursor() as cur:
            cur.execute(sql)
        conn.commit()
    finally:
        conn.close()

print("Helper functions defined. Ready for indexing practice!")

Helper functions defined. Ready for indexing practice!


### What is EXPLAIN ANALYZE?

`EXPLAIN ANALYZE` is PostgreSQL's built-in profiling tool. It does two things:

1. **EXPLAIN** — shows the query plan (how PostgreSQL *intends* to execute the query)
2. **ANALYZE** — actually *runs* the query and reports real execution times

The output is text, not a table, so we will print it directly rather than using the DataFrame return.

In [2]:
def run_explain(sql: str) -> None:
    """Run EXPLAIN ANALYZE and print the text output."""
    conn = psycopg2.connect(
        host="localhost", port=5432,
        dbname="week2_db", user="student", password="student123"
    )
    try:
        with conn.cursor() as cur:
            cur.execute(f"EXPLAIN ANALYZE {sql}")
            plan = cur.fetchall()
            for row in plan:
                print(row[0])
    finally:
        conn.close()

print("run_explain helper ready.")

run_explain helper ready.


In [3]:
# Reset: drop any practice indexes left over from a previous run of this notebook,
# so it can always be re-run top-to-bottom (idempotent — see Section 6!)
for idx in ["idx_employees_salary", "idx_employees_department", "idx_employees_dept_salary",
            "idx_employees_hire_date", "idx_employees_dept_active_salary"]:
    run_command(f"DROP INDEX IF EXISTS company.{idx}")
print("Practice indexes reset. Starting from a clean slate.")

Practice indexes reset. Starting from a clean slate.


Let's quickly check the employees table structure so we know what columns are available for indexing.

In [4]:
run_query("""
    SELECT column_name, data_type 
    FROM information_schema.columns 
    WHERE table_schema = 'company' AND table_name = 'employees'
    ORDER BY ordinal_position
""")

/tmp/ipykernel_118062/3666126203.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, conn)


,column_name,data_type
0,emp_id,integer
1,first_name,character varying
2,last_name,character varying
3,email,character varying
4,department_id,integer
5,salary,numeric
6,hire_date,date
7,manager_id,integer
8,is_active,boolean


**Expected output:** Column names like emp_id, first_name, last_name, email, department_id, salary, hire_date, manager_id, is_active with their data types. These are the columns we will experiment with.

In [5]:
# Check existing indexes on the employees table
run_query("""
    SELECT indexname, indexdef 
    FROM pg_indexes 
    WHERE schemaname = 'company' AND tablename = 'employees'
""")

/tmp/ipykernel_118062/3666126203.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, conn)


,indexname,indexdef
0,employees_pkey,CREATE UNIQUE INDEX employees_pkey ON company....
1,employees_email_key,CREATE UNIQUE INDEX employees_email_key ON com...


**Expected output:** Likely just the primary key index on `emp_id` (automatically created). No index on `salary` yet — that's what we'll add.

---

## 1. Sequential Scan — No Index

Let's start with a query that filters employees by salary, and see how PostgreSQL executes it *without* an index.

**Query:** Find all employees earning more than 80,000.

Without an index on `salary`, PostgreSQL must examine every single row to check if the condition is met. This is called a **Sequential Scan** (or "Seq Scan").

In [6]:
# EXPLAIN ANALYZE: SELECT with no index on salary
run_explain("""
    SELECT emp_id, first_name, last_name, salary
    FROM company.employees
    WHERE salary > 80000
""")

Seq Scan on employees  (cost=0.00..1.52 rows=14 width=256) (actual time=0.005..0.011 rows=18 loops=1)
  Filter: (salary > '80000'::numeric)
  Rows Removed by Filter: 24
Planning Time: 0.203 ms
Execution Time: 0.024 ms


**Expected output (annotated):**
```
Seq Scan on employees  (cost=0.00..1.50 rows=10 width=42) (actual time=0.010..0.015 rows=8 loops=1)
  Filter: (salary > 80000)
  Rows Removed by Filter: 42
Planning Time: 0.050 ms
Execution Time: 0.025 ms
```

**Key things to notice:**
- **`Seq Scan on employees`** — PostgreSQL reads every row from start to finish
- **`Filter: (salary > 80000)`** — it applies the WHERE condition as it scans
- **`Rows Removed by Filter: 42`** — 42 rows were checked and rejected (out of ~50 total)
- **`Execution Time: 0.025 ms`** — very fast because the table is tiny (~50 rows)

**Analogy:** A Seq Scan is like reading every page of a book to find mentions of a topic. Works fine for a 50-page book, but terrible for an encyclopedia.

---

## 2. Create an Index and Compare

Now let's create an index on the `salary` column. An index is like a sorted lookup table — instead of scanning all rows, PostgreSQL can jump directly to the relevant range.

**What happens internally:** PostgreSQL builds a B-tree index on `salary`, storing (salary_value, ctid) pairs in sorted order. When you query `WHERE salary > 80000`, it can binary-search the index to find matching row locations.

In [7]:
# Create the index (a DDL command — no rows returned, so we use run_command)
run_command("""
    CREATE INDEX idx_employees_salary ON company.employees(salary)
""")
print("Index created successfully!")

Index created successfully!


**Expected output:** `Index created successfully!` (The CREATE INDEX statement itself doesn't return rows — it's a DDL command.)

**What `CREATE INDEX` does:**
- Builds a B-tree structure sorted by `salary` values
- Stores a pointer (ctid) to each row's physical location
- The index is maintained automatically — INSERT, UPDATE, and DELETE will also update it
- **Cost:** Every write operation is now slightly slower because the index must be updated too

In [8]:
# Verify the index exists
run_query("""
    SELECT indexname, indexdef 
    FROM pg_indexes 
    WHERE schemaname = 'company' AND tablename = 'employees'
""")

/tmp/ipykernel_118062/3666126203.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, conn)


,indexname,indexdef
0,employees_pkey,CREATE UNIQUE INDEX employees_pkey ON company....
1,employees_email_key,CREATE UNIQUE INDEX employees_email_key ON com...
2,idx_employees_salary,CREATE INDEX idx_employees_salary ON company.e...


**Expected output:** Now you should see two indexes — the primary key on `emp_id` and our new `idx_employees_salary` on `salary`.

In [9]:
# EXPLAIN ANALYZE: Same query, now WITH an index on salary
run_explain("""
    SELECT emp_id, first_name, last_name, salary
    FROM company.employees
    WHERE salary > 80000
""")

Seq Scan on employees  (cost=0.00..1.52 rows=14 width=256) (actual time=0.006..0.011 rows=18 loops=1)
  Filter: (salary > '80000'::numeric)
  Rows Removed by Filter: 24
Planning Time: 0.248 ms
Execution Time: 0.023 ms


**Expected output (annotated):**
```
Bitmap Heap Scan on employees  (cost=4.30..12.50 rows=10 width=42) (actual time=0.015..0.020 rows=8 loops=1)
  Recheck Cond: (salary > 80000)
  Heap Blocks: exact=5
  ->  Bitmap Index Scan on idx_employees_salary  (cost=0.00..4.30 rows=10 width=0) (actual time=0.008..0.008 rows=8 loops=1)
        Index Cond: (salary > 80000)
Planning Time: 0.100 ms
Execution Time: 0.030 ms
```

**Key things to notice:**
- **`Bitmap Index Scan on idx_employees_salary`** — PostgreSQL used our new index! It found the matching rows via the index rather than scanning everything
- **`Bitmap Heap Scan`** — After the index identifies which rows match, PostgreSQL fetches those specific rows from the heap (the actual table storage)
- **`Index Cond: (salary > 80000)`** — the index was used for our exact condition
- **No `Rows Removed by Filter`** — because the index pre-filtered; no unnecessary rows were examined

**Important note about timing:** On a 50-row table, the indexed version might actually be *slower* in absolute milliseconds! This is because index lookups have overhead (reading the index tree + fetching rows). The benefit becomes dramatic at 50,000 or 50,000,000 rows.

**The real win:** At scale, a Seq Scan is O(n) — it grows linearly with table size. An Index Scan is O(log n) — it grows logarithmically. For a million rows, the index might check ~20 entries instead of 1,000,000.

---

## 3. Indexing Foreign Keys for JOINs

Indexes are especially valuable on **foreign key columns** used in JOINs. Without an index on `department_id`, every JOIN requires scanning the entire employees table.

Let's test a JOIN query before and after indexing `department_id`.

First, let's check if there's already an index on `department_id`.

In [10]:
# Check current indexes
run_query("""
    SELECT indexname, indexdef 
    FROM pg_indexes 
    WHERE schemaname = 'company' AND tablename = 'employees'
""")

/tmp/ipykernel_118062/3666126203.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, conn)


,indexname,indexdef
0,employees_pkey,CREATE UNIQUE INDEX employees_pkey ON company....
1,employees_email_key,CREATE UNIQUE INDEX employees_email_key ON com...
2,idx_employees_salary,CREATE INDEX idx_employees_salary ON company.e...


**Expected output:** You should see indexes for `emp_id` (primary key) and `salary` (the one we just created). There should NOT be one for `department_id` yet.

In [11]:
# EXPLAIN ANALYZE: JOIN before indexing department_id
run_explain("""
    SELECT e.first_name, e.last_name, d.dept_name, e.salary
    FROM company.employees e
    JOIN company.departments d ON e.department_id = d.dept_id
    WHERE d.dept_name = 'Engineering'
""")

Hash Join  (cost=12.01..13.54 rows=1 width=470) (actual time=0.040..0.046 rows=8 loops=1)
  Hash Cond: (e.department_id = d.dept_id)
  ->  Seq Scan on employees e  (cost=0.00..1.42 rows=42 width=256) (actual time=0.004..0.006 rows=42 loops=1)
  ->  Hash  (cost=12.00..12.00 rows=1 width=222) (actual time=0.013..0.013 rows=1 loops=1)
        Buckets: 1024  Batches: 1  Memory Usage: 9kB
        ->  Seq Scan on departments d  (cost=0.00..12.00 rows=1 width=222) (actual time=0.004..0.005 rows=1 loops=1)
              Filter: ((dept_name)::text = 'Engineering'::text)
              Rows Removed by Filter: 5
Planning Time: 0.454 ms
Execution Time: 0.073 ms


**Expected output (annotated):**
```
Nested Loop  (cost=0.00..2.50 rows=5 width=50) (actual time=0.015..0.025 rows=10 loops=1)
  ->  Seq Scan on departments d  (cost=0.00..1.05 rows=1 width=20) (actual time=0.005..0.006 rows=1 loops=1)
        Filter: (dept_name = 'Engineering'::text)
        Rows Removed by Filter: 5
  ->  Seq Scan on employees e  (cost=0.00..1.50 rows=10 width=38) (actual time=0.005..0.015 rows=10 loops=1)
        Filter: (e.department_id = d.dept_id)
        Rows Removed by Filter: 40
Planning Time: 0.200 ms
Execution Time: 0.035 ms
```

**Key things to notice:**
- **`Seq Scan on employees`** in the JOIN — every employee row is scanned to find matching department_id
- **`Rows Removed by Filter: 40`** — 40 employee rows were examined and rejected
- The departments table is tiny so its Seq Scan is fine. The employees table is the bottleneck.

**Problem:** For every department found in the outer loop, PostgreSQL scans ALL employees to find matches. This is O(n*m) without an index.

In [12]:
# Create an index on department_id
run_command("""
    CREATE INDEX idx_employees_department ON company.employees(department_id)
""")
print("Index on department_id created!")

Index on department_id created!


In [13]:
# EXPLAIN ANALYZE: Same JOIN after indexing department_id
run_explain("""
    SELECT e.first_name, e.last_name, d.dept_name, e.salary
    FROM company.employees e
    JOIN company.departments d ON e.department_id = d.dept_id
    WHERE d.dept_name = 'Engineering'
""")

Hash Join  (cost=12.01..13.54 rows=1 width=470) (actual time=0.029..0.036 rows=8 loops=1)
  Hash Cond: (e.department_id = d.dept_id)
  ->  Seq Scan on employees e  (cost=0.00..1.42 rows=42 width=256) (actual time=0.005..0.007 rows=42 loops=1)
  ->  Hash  (cost=12.00..12.00 rows=1 width=222) (actual time=0.012..0.013 rows=1 loops=1)
        Buckets: 1024  Batches: 1  Memory Usage: 9kB
        ->  Seq Scan on departments d  (cost=0.00..12.00 rows=1 width=222) (actual time=0.004..0.004 rows=1 loops=1)
              Filter: ((dept_name)::text = 'Engineering'::text)
              Rows Removed by Filter: 5
Planning Time: 0.489 ms
Execution Time: 0.063 ms


**Expected output (annotated):**
```
Nested Loop  (cost=0.00..12.50 rows=5 width=50) (actual time=0.010..0.020 rows=10 loops=1)
  ->  Seq Scan on departments d  (cost=0.00..1.05 rows=1 width=20) (actual time=0.004..0.005 rows=1 loops=1)
        Filter: (dept_name = 'Engineering'::text)
        Rows Removed by Filter: 5
  ->  Bitmap Heap Scan on employees e  (cost=0.15..11.50 rows=5 width=38) (actual time=0.005..0.010 rows=10 loops=1)
        Recheck Cond: (e.department_id = d.dept_id)
        Heap Blocks: exact=3
        ->  Bitmap Index Scan on idx_employees_department  (cost=0.00..0.15 rows=5 width=0) (actual time=0.003..0.003 rows=10 loops=1)
              Index Cond: (e.department_id = d.dept_id)
Planning Time: 0.250 ms
Execution Time: 0.030 ms
```

**Key changes:**
- **`Bitmap Index Scan on idx_employees_department`** — now PostgreSQL uses the index to directly find employees in the Engineering department
- **No `Rows Removed by Filter`** on employees — the index pre-filtered, so only matching rows were fetched
- At scale, this changes the JOIN from O(n*m) to O(n + log m), a massive improvement

**Best practice:** Always index your foreign key columns. They are used in JOINs constantly, and the write overhead is minimal compared to the read benefit.

---

## 4. Composite Indexes

A **composite index** (also called a multi-column index) covers two or more columns together. This is useful when queries filter on both columns, or on the *leftmost prefix* of the index.

**Key rule:** A composite index on `(A, B)` can be used for:
- Queries filtering on `A` alone
- Queries filtering on `A` and `B` together
- Queries filtering on `A` with a range on `B`

**But NOT for:**
- Queries filtering on `B` alone (the leftmost column must be involved)

Let's create a composite index on `(department_id, salary)` and test both scenarios.

In [14]:
# Create a composite index on (department_id, salary)
run_command("""
    CREATE INDEX idx_employees_dept_salary ON company.employees(department_id, salary)
""")
print("Composite index created on (department_id, salary)!")

Composite index created on (department_id, salary)!


**How the composite index works:**
- The index is sorted first by `department_id`, then by `salary` within each department
- Think of it like a phone book sorted by last name, then first name
- You can efficiently look up "all Smiths" (department_id alone) or "John Smith" (both columns)
- But you CANNOT efficiently look up "everyone named John" (salary alone without department_id)

In [15]:
# Query that CAN use the composite index: filters on both department_id AND salary
run_explain("""
    SELECT emp_id, first_name, last_name, department_id, salary
    FROM company.employees
    WHERE department_id = 1 AND salary > 70000
""")

Seq Scan on employees  (cost=0.00..1.63 rows=1 width=260) (actual time=0.005..0.010 rows=7 loops=1)
  Filter: ((salary > '70000'::numeric) AND (department_id = 1))
  Rows Removed by Filter: 35
Planning Time: 0.319 ms
Execution Time: 0.023 ms


**Expected output (annotated):**
```
Bitmap Heap Scan on employees  (cost=4.30..12.50 rows=5 width=42) (actual time=0.010..0.015 rows=4 loops=1)
  Recheck Cond: ((department_id = 1) AND (salary > 70000))
  Heap Blocks: exact=2
  ->  Bitmap Index Scan on idx_employees_dept_salary  (cost=0.00..4.30 rows=5 width=0) (actual time=0.005..0.005 rows=4 loops=1)
        Index Cond: ((department_id = 1) AND (salary > 70000))
Planning Time: 0.150 ms
Execution Time: 0.025 ms
```

**Key observation:** `Bitmap Index Scan on idx_employees_dept_salary` — the composite index is used because the query filters on `department_id` (the leftmost column) AND also on `salary`. Both conditions are satisfied by the index's sort order.

In [16]:
# Query that CANNOT efficiently use the composite index: filters on salary alone
run_explain("""
    SELECT emp_id, first_name, last_name, department_id, salary
    FROM company.employees
    WHERE salary > 90000
""")

Seq Scan on employees  (cost=0.00..1.52 rows=14 width=260) (actual time=0.005..0.011 rows=12 loops=1)
  Filter: (salary > '90000'::numeric)
  Rows Removed by Filter: 30
Planning Time: 0.318 ms
Execution Time: 0.025 ms


**Expected output (annotated):**
```
Bitmap Heap Scan on employees  (cost=4.30..12.50 rows=5 width=42) (actual time=0.010..0.015 rows=3 loops=1)
  Recheck Cond: (salary > 90000)
  ->  Bitmap Index Scan on idx_employees_salary  (cost=0.00..4.30 rows=5 width=0) (actual time=0.005..0.005 rows=3 loops=1)
        Index Cond: (salary > 90000)
Planning Time: 0.120 ms
Execution Time: 0.020 ms
```

**Key observation:** PostgreSQL uses `idx_employees_salary` (the single-column index), NOT `idx_employees_dept_salary`. The composite index's leftmost column (`department_id`) is not in the WHERE clause, so it cannot be used efficiently.

**Why this matters:** The composite index is sorted by department_id first. Without knowing the department_id, PostgreSQL would still need to scan the entire index — which is no better than scanning the table itself. The optimizer correctly falls back to the single-column salary index.

**Rule of thumb:** When designing composite indexes, put the most selective (most often filtered) column first. If queries frequently filter on `department_id` alone and on `(department_id, salary)` together, a composite index `(department_id, salary)` covers both cases.

---

## 5. When the Optimizer Ignores Your Index

Here is a crucial lesson: **PostgreSQL's query optimizer is smart**. It considers the cost of using an index vs. a sequential scan and picks whichever it estimates will be faster.

For **small tables** (like our ~50-row employees table), a sequential scan is often *faster* than an index scan, even when an index exists. This is because:

1. **Sequential scans are highly optimized** — they read contiguous blocks of memory/disk, which is very fast
2. **Index scans have overhead** — they require reading the index tree structure, then jumping to scattered row locations (random I/O)
3. **For tiny tables**, scanning 50 rows is nearly instantaneous; the index lookup overhead actually makes it slower

This is **correct behavior**. The optimizer knows the table is small and makes the right call.

Let's demonstrate: run a query that selects a large percentage of the table.

In [17]:
# Query that the optimizer may choose Seq Scan for (even with an index)
# because it returns most of the table
run_explain("""
    SELECT emp_id, first_name, last_name, salary
    FROM company.employees
    WHERE salary > 30000
""")

Seq Scan on employees  (cost=0.00..1.52 rows=14 width=256) (actual time=0.005..0.012 rows=42 loops=1)
  Filter: (salary > '30000'::numeric)
Planning Time: 0.289 ms
Execution Time: 0.026 ms


**Expected output (annotated):**
```
Seq Scan on employees  (cost=0.00..1.50 rows=48 width=42) (actual time=0.005..0.010 rows=48 loops=1)
  Filter: (salary > 30000)
  Rows Removed by Filter: 2
Planning Time: 0.050 ms
Execution Time: 0.015 ms
```

**What happened:** Even though `idx_employees_salary` exists, PostgreSQL chose a `Seq Scan`. Why?

- The query `salary > 30000` matches ~48 out of 50 rows (almost the entire table)
- Using the index would require: reading the index + fetching 48 scattered rows from the heap
- A sequential scan just reads the table in one pass — faster for this many rows

**The tipping point:** PostgreSQL typically switches to an index scan when the query returns less than ~5-10% of the table. When you need 96% of rows, a Seq Scan wins.

**This is not a bug — it's the optimizer doing its job correctly.** On a table with 5,000,000 rows, the same query would almost certainly use the index (because scanning 5 million rows sequentially is genuinely expensive).

**Takeaway:** Don't panic if EXPLAIN ANALYZE shows "Seq Scan" on a small table even with an index. The optimizer is making the right choice. Indexes shine on large tables.

---

## 6. Dropping Indexes

Indexes are not free. They:
- Consume disk space
- Slow down INSERT, UPDATE, and DELETE (every write must update the index)
- Add overhead to the query planner (more indexes = more plans to consider)

When an index is no longer useful, you should drop it. Use `DROP INDEX` to remove one.

In [18]:
# List all current indexes on employees
run_query("""
    SELECT indexname, indexdef 
    FROM pg_indexes 
    WHERE schemaname = 'company' AND tablename = 'employees'
""")

/tmp/ipykernel_118062/3666126203.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, conn)


,indexname,indexdef
0,employees_pkey,CREATE UNIQUE INDEX employees_pkey ON company....
1,employees_email_key,CREATE UNIQUE INDEX employees_email_key ON com...
2,idx_employees_salary,CREATE INDEX idx_employees_salary ON company.e...
3,idx_employees_department,CREATE INDEX idx_employees_department ON compa...
4,idx_employees_dept_salary,CREATE INDEX idx_employees_dept_salary ON comp...


**Expected output:** You should now see several indexes:
- `employees_pkey` (primary key on emp_id, auto-created)
- `idx_employees_salary` (single-column index)
- `idx_employees_department` (single-column index)
- `idx_employees_dept_salary` (composite index)

In [19]:
# Drop the single-column salary index (we have the composite one)
run_command("""
    DROP INDEX company.idx_employees_salary
""")
print("Dropped idx_employees_salary")

Dropped idx_employees_salary


**Why drop it?** The composite index `idx_employees_dept_salary` starts with `department_id`, not `salary`. So for queries that filter only on `salary`, the composite index is useless. If we had queries that filter on `salary` alone frequently, we'd keep the single-column index. But if most salary queries also include `department_id`, the composite index covers it and we can save disk space by dropping the redundant one.

**Note:** We use `company.idx_employees_salary` (schema-qualified) because the index lives in the `company` schema. Without the schema prefix, PostgreSQL might not find it.

In [20]:
# Verify: index is gone
run_query("""
    SELECT indexname, indexdef 
    FROM pg_indexes 
    WHERE schemaname = 'company' AND tablename = 'employees'
""")

/tmp/ipykernel_118062/3666126203.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, conn)


,indexname,indexdef
0,employees_pkey,CREATE UNIQUE INDEX employees_pkey ON company....
1,employees_email_key,CREATE UNIQUE INDEX employees_email_key ON com...
2,idx_employees_department,CREATE INDEX idx_employees_department ON compa...
3,idx_employees_dept_salary,CREATE INDEX idx_employees_dept_salary ON comp...


**Expected output:** `idx_employees_salary` should no longer appear. The composite index and department index should remain.

In [21]:
# DROP INDEX IF EXISTS — safe version that won't error if the index is already gone
run_command("""
    DROP INDEX IF EXISTS company.idx_employees_salary
""")
print("Safe drop complete — no error even if the index doesn't exist")

Safe drop complete — no error even if the index doesn't exist


**Best practice:** Use `DROP INDEX IF EXISTS` in scripts and migrations. It won't throw an error if the index has already been removed, making your scripts idempotent (safe to run multiple times).

---

## 7. Try It Yourself Exercises

Now it's your turn. Attempt each exercise before revealing the solution.

---

### Exercise 1: Index on hire_date

Create an index on `company.employees(hire_date)`, then use `EXPLAIN ANALYZE` to compare the query plan before and after for this query:

```sql
SELECT emp_id, first_name, last_name, hire_date
FROM company.employees
WHERE hire_date > '2020-01-01'
```

What scan type does PostgreSQL use before the index? What about after? Does the execution time change noticeably?

<details>
<summary><strong>Click to reveal solution</strong></summary>

**Step 1: Check the plan BEFORE creating the index**

```sql
EXPLAIN ANALYZE
SELECT emp_id, first_name, last_name, hire_date
FROM company.employees
WHERE hire_date > '2020-01-01';
```

**Expected:** `Seq Scan on employees` — no index on hire_date exists yet.

**Step 2: Create the index**

```sql
CREATE INDEX idx_employees_hire_date ON company.employees(hire_date);
```

**Step 3: Check the plan AFTER creating the index**

```sql
EXPLAIN ANALYZE
SELECT emp_id, first_name, last_name, hire_date
FROM company.employees
WHERE hire_date > '2020-01-01';
```

**Expected:** You may see `Bitmap Index Scan on idx_employees_hire_date` or `Seq Scan` depending on how many rows match the condition. If most employees were hired after 2020, the optimizer may still choose Seq Scan (see Section 5). Both outcomes are correct.

**Step 4: Clean up**

```sql
DROP INDEX IF EXISTS company.idx_employees_hire_date;
```
</details>

---

### Exercise 2: Find the Right Composite Index

Design a composite index that would optimize this query. Explain why you chose the column order. Then create it, run EXPLAIN ANALYZE, and verify the index is used.

```sql
SELECT emp_id, first_name, last_name, department_id, salary, is_active
FROM company.employees
WHERE department_id = 2 AND is_active = true
ORDER BY salary DESC
```

**Hint:** Think about which columns are used in equality conditions (=), which are used in ORDER BY, and how the index's sort order can help.

<details>
<summary><strong>Click to reveal solution</strong></summary>

**Analysis:**
- The query filters on `department_id = 2` (equality) and `is_active = true` (equality)
- It sorts by `salary DESC`
- A good composite index would be `(department_id, is_active, salary)`

**Why this column order?**
1. `department_id` first — it's an equality filter, and it reduces the search space most (there are ~6 departments, so this filters to ~1/6 of rows)
2. `is_active` second — another equality filter, further narrowing results within each department
3. `salary` last — it's used in ORDER BY. Having it as the last column in the index means rows are already sorted by salary within each (department_id, is_active) group, so PostgreSQL can avoid a separate sort step

**Create the index:**

```sql
CREATE INDEX idx_employees_dept_active_salary 
ON company.employees(department_id, is_active, salary);
```

**Verify with EXPLAIN ANALYZE:**

```sql
EXPLAIN ANALYZE
SELECT emp_id, first_name, last_name, department_id, salary, is_active
FROM company.employees
WHERE department_id = 2 AND is_active = true
ORDER BY salary DESC;
```

**Expected output:** `Bitmap Index Scan on idx_employees_dept_active_salary` — the index is used. You might also notice `Sort` is absent (or very cheap) because the index already provides the salary ordering within the filtered group.

**Clean up:**

```sql
DROP INDEX IF EXISTS company.idx_employees_dept_active_salary;
```

**Bonus insight:** This three-column index is a "covering index" for this query pattern — it contains all the columns needed for filtering, and its sort order helps with ORDER BY. However, it won't help queries that filter on `is_active` alone (leftmost column rule).
</details>

---

## Summary: Indexing Rules of Thumb

| Situation | Recommendation |
|-----------|---------------|
| Primary key | Auto-indexed; no action needed |
| Foreign key | **Always** create an index |
| Column used in WHERE with high selectivity (few matches) | Create an index |
| Column used in WHERE with low selectivity (most rows match) | Index may not be used; Seq Scan might be better |
| Multiple columns frequently filtered together | Consider a composite index |
| Composite index column order | Most selective/equality columns first, range/sort columns last |
| Small table (< 1000 rows) | Optimizer may prefer Seq Scan; don't worry about it |
| Write-heavy table | Be selective with indexes; each one slows down writes |
| Unused indexes | Drop them; they waste space and slow writes |

**The golden rule:** Always use `EXPLAIN ANALYZE` to verify your assumptions. The optimizer's actual behavior may differ from what you expect — and that's okay. Trust the data, not the intuition.